In [1]:
import osmnx as ox
import networkx as nx
import logging
from typing import Tuple, Optional
import matplotlib.pyplot as plt

In [2]:
# ----------------------------------------------------------------------
# Configuration (easily adjustable)
# ----------------------------------------------------------------------
PLACE_NAME = "Maarif, Casablanca, Morocco"
NETWORK_TYPE = "drive"          # 'drive', 'walk', 'bike', 'all'
AMENITY_TAG = {"amenity": "cafe"}
GRAPH_SAVE_PATH = "maarif_graph.graphml"   # to cache downloads
ROUTE_COLOR = "red"
ROUTE_LINEWIDTH = 4
BG_COLOR = "#111111"

# Set up logging (more informative than print)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# ----------------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------------
def load_or_download_graph(place: str, network_type: str, save_path: str) -> nx.MultiDiGraph:
    """Load graph from disk if exists, otherwise download and cache."""
    try:
        G = ox.load_graphml(save_path)
        logger.info(f"Graph loaded from cache: {save_path}")
    except FileNotFoundError:
        logger.info(f"Downloading graph for {place} (network_type={network_type})...")
        G = ox.graph_from_place(place, network_type=network_type)
        ox.save_graphml(G, save_path)
        logger.info(f"Graph saved to {save_path}")
    return G

def ensure_graph_is_connected(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """Keep only the largest strongly connected component (for routing)."""
    if not nx.is_strongly_connected(G):
        # For directed graphs (drive network), we need strongly connected
        # If you prefer undirected, use G = G.to_undirected()
        largest_scc = max(nx.strongly_connected_components(G), key=len)
        G = G.subgraph(largest_scc).copy()
        logger.warning(f"Graph was disconnected. Kept largest SCC with {G.number_of_nodes()} nodes.")
    else:
        logger.info("Graph is strongly connected.")
    return G

def get_nearest_nodes_from_place(G: nx.MultiDiGraph, place: str) -> Tuple[int, int]:
    """
    Get two distinct nodes inside the graph by querying the OSM place polygon
    and snapping random points to the graph.
    """
    # Get the boundary polygon of the place
    area = ox.geocode_to_gdf(place)
    if area.empty:
        raise ValueError(f"Could not geocode {place}")
    polygon = area.geometry.iloc[0]
    # Sample two random points inside the polygon
    points = ox.utils_geo.sample_points(polygon, 2)
    nodes = []
    for pt in points:
        node = ox.distance.nearest_nodes(G, pt.x, pt.y)
        nodes.append(node)
    # Ensure they are distinct (if not, shift slightly)
    if nodes[0] == nodes[1]:
        logger.warning("Random points snapped to same node. Retrying...")
        return get_nearest_nodes_from_place(G, place)   # recursion (safe depth 1)
    return nodes[0], nodes[1]

def compute_shortest_route(G: nx.MultiDiGraph, orig_node: int, dest_node: int, weight: str = "length"):
    """Compute shortest path; raise exception if none exists."""
    try:
        route = nx.shortest_path(G, orig_node, dest_node, weight=weight)
        logger.info(f"Shortest path found with {len(route)} nodes.")
        return route
    except nx.NetworkXNoPath:
        logger.error(f"No path between nodes {orig_node} and {dest_node}.")
        raise

def plot_route(G: nx.MultiDiGraph, route: list, route_color: str, route_linewidth: int, bg_color: str):
    """Plot the graph with the highlighted route."""
    fig, ax = ox.plot_graph_route(
        G, route,
        route_color=route_color,
        route_linewidth=route_linewidth,
        node_size=0,
        bgcolor=bg_color,
        show=False,
        close=False
    )
    plt.title(f"Shortest path – Length: {compute_route_length(G, route):.0f} m")
    plt.show()

def compute_route_length(G: nx.MultiDiGraph, route: list) -> float:
    """Return total length of the route in meters."""
    route_gdf = ox.routing.route_to_gdf(G, route, weight="length")
    return route_gdf['length'].sum()

# ----------------------------------------------------------------------
# Main workflow
# ----------------------------------------------------------------------
def main():
    # 1. Load or download graph
    G = load_or_download_graph(PLACE_NAME, NETWORK_TYPE, GRAPH_SAVE_PATH)
    logger.info(f"Graph has {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

    # 2. Ensure connectivity (critical to avoid NetworkXNoPath)
    G = ensure_graph_is_connected(G)

    # 3. Fetch POIs (cafes) – just for demonstration
    try:
        cafes = ox.features_from_place(PLACE_NAME, tags=AMENITY_TAG)
        logger.info(f"Found {len(cafes)} cafes in {PLACE_NAME}.")
    except Exception as e:
        logger.warning(f"Could not fetch cafes: {e}")
        cafes = None

    # 4. Choose origin and destination nodes intelligently (no random node list)
    origin_node, destination_node = get_nearest_nodes_from_place(G, PLACE_NAME)
    logger.info(f"Origin node: {origin_node}, Destination node: {destination_node}")

    # 5. Compute shortest route
    route = compute_shortest_route(G, origin_node, destination_node, weight="length")
    route_length = compute_route_length(G, route)
    logger.info(f"Route length: {route_length:.2f} meters")

    # 6. Visualize
    plot_route(G, route, ROUTE_COLOR, ROUTE_LINEWIDTH, BG_COLOR)

if __name__ == "__main__":
    main()

2026-04-23 20:31:01,334 - INFO - Downloading graph for Maarif, Casablanca, Morocco (network_type=drive)...
2026-04-23 20:31:03,319 - INFO - Graph saved to maarif_graph.graphml
2026-04-23 20:31:03,321 - INFO - Graph has 1934 nodes, 4531 edges.
2026-04-23 20:31:03,359 - WARNING - Graph was disconnected. Kept largest SCC with 1909 nodes.
2026-04-23 20:31:03,447 - INFO - Found 85 cafes in Maarif, Casablanca, Morocco.


AttributeError: 'Polygon' object has no attribute 'is_directed'